# Data Analysis Workshop

## What you will learn

By the end of the workshop, you will be able to:

- load a survey, select the answers you need, and prepare a table for analysis;
- identify missing values and count individual responses;
- create and describe bar charts, change their colors, and display counts or percentages;
- compare group responses in a chart and explain how many people each percentage is based on;
- compare the meanings of words and comments using one positive and one negative example;
- mask email addresses and phone numbers using regex patterns and see how a model recognizes a person or place from context.

## How to use this notebook

There are **8 exercises**. Each has instructions, a hint, an example, and a cell to complete. Run the code in order. If you lose track, use the [answer notebook](https://colab.research.google.com/github/bajon1/Data-Analysis-Workshop/blob/main/notebooks/workshop_solutions.ipynb).

## Our data

The [2014 OSMI survey](https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey) concerns mental health in the technology industry. We select six questions about work and support. Personal information and comments are excluded from the exercises. Results describe this survey, not diagnoses or the whole population.


In [ ]:
# Run this cell once to make the workshop helper modules available.
import io
import os
import sys
import zipfile
from pathlib import Path
from urllib.request import urlopen

# Download all repository files, including helper modules, once per Colab session.
if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/Data-Analysis-Workshop-main")
    if not REPO_DIR.is_dir():
        archive_url = "https://github.com/bajon1/Data-Analysis-Workshop/archive/refs/heads/main.zip"
        with urlopen(archive_url, timeout=60) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(REPO_DIR.parent)
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == "notebooks":
        REPO_DIR = REPO_DIR.parent

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")


## Part 0: Setup — opening the survey

Each row contains one person's answers, and each column represents one question.

Run the code below using the button on the left. It downloads `survey.csv` and opens it as a table called **df_raw**. You do not need to download the file manually first.

The column dictionary is below the code. We will select six questions for the exercises and call this smaller table **df**.


In [ ]:
%pip -q install kagglehub

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

data_file = kagglehub.dataset_download(
    "osmi/mental-health-in-tech-survey",
    path="survey.csv",
)
df_raw = pd.read_csv(data_file)
print("Survey responses:", len(df_raw))


### Source file column dictionary

<details>
<summary>Expand the full dictionary — reference material to consult when needed</summary>

Descriptions follow the [authors' dataset page](https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey).

| Column | Description |
|---|---|
| `Timestamp` | Time the survey response was submitted. |
| `Age` | Age. |
| `Gender` | Gender. |
| `Country` | Country. |
| `state` | State or territory, if the respondent lives in the United States. |
| `self_employed` | Are you self-employed? |
| `family_history` | Do you have a family history of mental illness? |
| `treatment` | Have you sought treatment for a mental health condition? |
| `work_interfere` | If you have a mental health condition, does it interfere with your work? |
| `no_employees` | How many employees does your company or organization have? |
| `remote_work` | Do you work remotely, outside an office, at least 50% of the time? |
| `tech_company` | Is your employer primarily a technology company or organization? |
| `benefits` | Does your employer provide mental health benefits? |
| `care_options` | Do you know the mental health care options your employer provides? |
| `wellness_program` | Has your employer discussed mental health as part of an employee wellness program? |
| `seek_help` | Does your employer provide resources about mental health and seeking help? |
| `anonymity` | Is your anonymity protected when using mental health or substance use support? |
| `leave` | How easy is it to take medical leave for a mental health condition? |
| `mental_health_consequence` | Could discussing mental health with your employer have negative consequences? |
| `phys_health_consequence` | Could discussing physical health with your employer have negative consequences? |
| `coworkers` | Would you be willing to discuss mental health with coworkers? |
| `supervisor` | Would you be willing to discuss mental health with your direct supervisor? |
| `mental_health_interview` | Would you bring up mental health in a job interview? |
| `phys_health_interview` | Would you bring up physical health in a job interview? |
| `mental_vs_physical` | Does your employer take mental health as seriously as physical health? |
| `obs_consequence` | Have you heard of or observed negative consequences for coworkers with mental health conditions? |
| `comments` | Additional free-text comments. We do not use them in this course. |

`treatment` means seeking treatment, not a diagnosis. Selecting a few columns does not by itself ensure anonymity.

</details>


### Our working table

We work with six questions:

| Column | What is it about? |
|---|---|
| `remote_work` | Working remotely at least half the time. |
| `self_employed` | Self-employment. |
| `benefits` | Employer-provided mental health benefits. |
| `care_options` | Awareness of employer-provided care options. |
| `wellness_program` | Mental health discussions in the company's wellness program. |
| `seek_help` | Employer resources about mental health and seeking help. |

Response categories include **Yes**, **No**, **Don't know**, and **Not sure**. We keep these distinctions.

The code below creates the table `df`. We will explain the brackets in Part 1.


In [ ]:
workshop_columns = [
    "remote_work",
    "self_employed",
    "benefits",
    "care_options",
    "wellness_program",
    "seek_help",
]

df = df_raw[workshop_columns].copy()
df.head()


## Part 1: Reading short commands

### What to do, with what, and how much to show?

`df` is the name of our table. Whenever it appears in the code, it refers to that table.

Look at this expression:

```python
df.head(3)
```

Read it as: **"Show the first 3 rows of the table df."**

| Part | What does it mean here? |
|---|---|
| `df` | Our table. |
| `.` | What follows the dot applies to this table. |
| `head` | Show the beginning of the table: its first rows. |
| `(3)` | Show 3 rows. Put the number between the opening `(` and closing `)` parentheses. |

Changing `3` to `2` changes the number of rows displayed. The table itself stays the same.

Read the other expressions in the next cell as follows:

| Expression | Result |
|---|---|
| `len(df)` | Counts the rows. |
| `print("Hello")` | Displays Hello. Quotation marks indicate text. |
| `df.head()` | The first 5 rows: the default when no number is specified. |
| `df.shape` | Gives the numbers of rows and columns; no `()` because this is a known property of the table. |

In `len(df)`, we start with what to do — count — and put what we are counting, `df`, in parentheses. In `df.head(3)`, we start with the table and put what to do with it after the dot.

Parentheses `()` appear in commands to execute, but their contents depend on the command: `len(df)` takes a table, `head(3)` takes a number of rows, and `print("Hello")` takes text to display.

We can also combine familiar elements:

```python
print("Number of rows:", len(df))
```

First, `len(df)` counts the rows. Then `print(...)` displays "Number of rows:" and the calculated number. The comma separates the two things to display. The inner closing parenthesis ends the count; the outer one ends the whole display command.

One more useful expression:

```python
count = len(df)
```

Read it as: **"Count the rows in df and store the result under the name count."** A single `=` assigns a name to a value. Later, `print(count)` displays the stored result.


In [ ]:
print("Number of rows:", len(df))
print("Rows and columns:", df.shape)
df.head(3)


### Selecting one column or several

**By name:**

```python
df["benefits"]
```

Read it as: **"Select the benefits column from the table df."** The text in quotes is the exact column heading. Pandas finds the column by name; you do not need its number. Spelling and capitalization must match.

**Several columns:**

```python
df[["benefits", "care_options"]]
```

| Brackets | Purpose |
|---|---|
| Inner: `["benefits", "care_options"]` | A list of names separated by a comma. |
| Outer: `df[...]` | Select the columns in that list. |

The order of names determines the order of columns in the result.

### By position: iloc

Positions start **at zero**:

| Position | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| Column | remote_work | self_employed | benefits | care_options | wellness_program | seek_help |

```python
df.iloc[:, [2, 3]]
```

These are again **benefits** and **care_options**. Before the comma, `:` means all rows; after the comma, provide column numbers without quotation marks.

**A range of adjacent columns:**

```python
df.iloc[:, 0:3]
```

`0:3` means **"from 0 up to, but excluding, position 3"**: positions **0, 1, 2**. For separate positions, use a list such as `[0, 4]`.

Selecting by position depends on column order. Be careful: `df[0:3]` alone selects **rows**; use `iloc[:, ...]` for columns by position.


In [ ]:
display(df[["benefits", "care_options"]].head(3))
display(df.iloc[:, [2, 3]].head(3))
df.iloc[:, 0:3].head(3)


### Exercise 1: select two questions

Show the first **5** rows of **remote_work** and **wellness_program**, in that order. Do this twice: select by column names and by column positions. Then show the first 5 rows of the first three columns using the range `0:3`.

**Hint:** Keep the double brackets when using names. For positions, use `iloc`, all rows (`:`), and a list of two numbers from the table above. A range excludes its final number.


In [ ]:
# Task: select remote_work and wellness_program; show 5 rows each.
# 1. Display the two selected columns

# 2. Change the range to the first three columns and show 5 rows.



### Selecting rows by asking a question

We want to keep people who work remotely:

```python
df["remote_work"] == "Yes"
```

Read it as: **"Is the answer in remote_work equal to Yes?"** The `==` operator compares both sides separately for each row.

A filter acts like a gate that lets matching responses through:

| Row (example) | remote_work | Is it Yes? | Filter |
|---|---|---|---|
| A | No | `False` — no | Skip. |
| B | Yes | `True` — yes | Keep. |
| C | Yes | `True` — yes | Keep. |
| D | No | `False` — no | Skip. |

**Do not confuse these two expressions:**

| Expression | Meaning |
|---|---|
| `count = 3` | Store 3 under the name count. |
| `count == 3` | Is count equal to 3? Result: True or False. |

```python
works_remotely = df["remote_work"] == "Yes"
remote_workers = df[works_remotely]
remote_workers.head(3)
```

1. Ask the question and store the True/False answers.
2. Keep only the rows in `df` marked True.
3. Show the first 3 matching rows.


In [ ]:
works_remotely = df["remote_work"] == "Yes"
print("Answers to the question for the first 5 rows:")
print(works_remotely.head(5))
remote_workers = df[works_remotely]
remote_workers.head(3)


### Exercise 2: preview support for remote workers

You want to examine benefits and awareness of care options among remote workers. Select people whose `remote_work` is **Yes**, print the size of this entire group, and show the first **4** rows of just **benefits** and **care_options**.

**Hint:** Combine four operations: a question using `==`, row selection, column selection, and a `head` preview. Use `len` to count the group before limiting the preview to 4 rows.


In [ ]:
# Task: preview benefits and care_options for remote workers.
# Select the entire group, count its rows, then show 4 rows of the two columns.
# Write your own code below.


### Exercise 3: examine the group unaware of benefits

You want to plan communication about support. Select people who answered **Don't know** in **benefits**. Print the number of people in this group, then show the first **6** rows of **remote_work** and **seek_help**. Is this preview alone enough to describe all the group's responses?

**Hint:** Select the group using one column, then show two other columns. Put `"Don't know"` in double quotes. `head(6)` shows only part of the group.

**Example to adapt:**

```python
matching = df["care_options"] == "Not sure"
selected = df[matching]
print("Number of people in the group:", len(selected))
selected[["remote_work", "wellness_program"]].head(2)
```


In [ ]:
# Task: select Don't know in benefits, count the group, and show 6 rows.
# Keep only remote_work and seek_help. Add a brief conclusion in a comment.
# Write your own code below.


## Part 2: Cleaning — checking missing values

### Counting missing responses

**No, Don't know, and Not sure are recorded responses, not missing values.** An empty entry may appear as `NaN`.

Earlier, `== "Yes"` asked: **"Is this Yes?"** Now, `isna()` asks: **"Is a response missing here?"**

| Example value | `== "Yes"` | `isna()` |
|---|---|---|
| Yes | True | False |
| No | False | False |
| Don't know / Not sure | False | False |
| missing (NaN) | False | True |

```text
responses:   Yes     [empty]   No     [empty]
isna():      False   True      False  True
sum():       0   +   1     +   0  +   1     = 2 missing
```

When summing, True counts as 1 and False as 0. Therefore, **`df.isna().sum()` finds missing values and counts them in each column**.

Use `isna()`, not `== "NaN"`: text in quotes does not represent an empty entry.


In [ ]:
missing_answers = df.isna().sum()
missing_answers


### Labeling missing values without guessing answers

`pd.Series(...)` creates one column, and `None` represents a missing value here.

`fillna("Not provided")` inserts a label only in empty entries. It does not change Yes or No and does not guess an answer.

`copy()` creates a separate copy of the table. Store the replacement result using `df_clean = df_clean.fillna(...)`.


In [ ]:
example_answers = pd.Series(["Yes", None, "No"])
example_answers.fillna("Not provided")


### Exercise 4: label missing responses

In the prepared copy **df_clean**, replace only empty entries with **"No response"**. Store the result under the same name and show the first **3** rows.

**Hint:** Start with `df_clean`, add `.fillna(...)`, and assign the result using `df_clean = ...`. Do not replace No or Don't know responses.

**Example to adapt:**

```python
example_answers = pd.Series(["Yes", None, "No"])
example_answers.fillna("Not provided")
```


In [ ]:
df_clean = df.copy()

# Task: in df_clean, label missing values as "No response" and store the result.
df_clean
# Show the first 3 rows.
df_clean.head(3)


### Checking the replacement with value_counts()

`value_counts()` shows **each response and how often it occurs**. Adding `dropna=False` also shows any remaining missing values.

Compare **self_employed before and after replacement**: the No response count should match the previous NaN count, while Yes and No remain unchanged. You will also see any other text values; `fillna()` changes only empty entries.

The code below recreates the shared table after Exercise 4. `display(...)` lets us show two summaries in one cell.

Identical sets of answers may come from different people; we do not remove them.


In [ ]:
df_clean = df.copy()
df_clean = df_clean.fillna("No response")

print("Before replacement:")
display(df["self_employed"].value_counts(dropna=False))
print("After replacement:")
display(df_clean["self_employed"].value_counts(dropna=False))


### Exercise 5: prepare a report summary

Using the original table **df**, prepare a report about remote workers:

1. Select `remote_work == "Yes"` and copy **self_employed** and **seek_help** into `remote_report`.
2. Print the number of people and missing values in both columns.
3. Label missing values **No response** and show the first **5** rows.
4. Count the categories of both questions, including any missing values. Add a one-sentence conclusion.

**Hint:** Combine filtering with `copy()`, `isna().sum()`, `fillna(...)`, and `value_counts(dropna=False)`. Count the entire group; keep `df_clean` for later work.

**Example to adapt:**

```python
matching = df["remote_work"] == "No"
example_report = df[matching][["benefits", "care_options"]].copy()
print("People in the group:", len(example_report))
print(example_report.isna().sum())
example_report = example_report.fillna("Not provided")
display(example_report.head(2))
example_report["benefits"].value_counts(dropna=False)
```


In [ ]:
# Task: prepare remote_report following the four steps above.
# Group and column selection:
# Number of people and missing values before replacement:
# Missing value labels and preview:
# Categories for both questions and a brief conclusion:
# Write your own code below.


## Part 3: Charts and comparisons

### A bar chart

Our question: **How do respondents answer the question about employer benefits?**

`plot.barh()` draws horizontal bars: their length represents the number of responses. The `plt` commands add labels.

Keep Don't know: being unaware of a benefit does not mean the benefit is unavailable.


In [ ]:
benefit_counts = df_clean["benefits"].value_counts()
benefit_counts.plot.barh(color="#356b8e")
plt.xlabel("Number of responses")
plt.title("Reported mental health benefits")
plt.show()


### Exercise 6: adapt a chart

Draw a horizontal bar chart of response counts for **wellness_program**. Give it a title that identifies the question.

**Hint:** Change the column and title. The variable name `counts` can stay the same. Keep all response categories.

**Example to adapt:**

```python
counts = df_clean["benefits"].value_counts()
counts.plot.barh(color="#356b8e")
plt.xlabel("Number of responses")
plt.title("Reported employer benefits")
plt.show()
```


In [ ]:
# Task: plot wellness_program counts and adapt the title.
counts = df_clean["benefits"].value_counts()
counts.plot.barh(color="#356b8e")
plt.xlabel("Number of responses")
plt.title("Reported employer benefits")
plt.show()


### Counts and percentages

`value_counts(normalize=True) * 100` gives percentages, and `round(1)` rounds to one decimal place.

The denominator is **all rows in df_clean**, including Don't know and No response. Results describe this survey; respondents volunteered, so they do not represent all employees.


In [ ]:
benefit_percent = df_clean["benefits"].value_counts(normalize=True) * 100
print("Denominator:", len(df_clean), "responses")
benefit_percent.round(1)


### Comparing groups

**Do benefits responses differ between remote workers and other respondents?**

`pd.crosstab(...)` counts combinations of responses: the first question determines the table's rows, and the second determines its columns. Check group sizes first.


In [ ]:
benefits_by_remote_counts = pd.crosstab(
    df_clean["remote_work"],
    df_clean["benefits"],
)
benefits_by_remote_counts


Adding `normalize="index"` and multiplying by 100 converts each group to percentages: each row adds up to **100%**.

Below, we inspect the table; in Exercise 7, you will draw a chart. `stacked=True` stacks the categories into one bar per group. Also check the counts: percentages from small groups may be unstable.


In [ ]:
benefits_by_remote_percent = pd.crosstab(
    df_clean["remote_work"],
    df_clean["benefits"],
    normalize="index",
) * 100

benefits_by_remote_percent.round(1)


### Exercise 7: combine familiar steps

Compare **wellness_program** responses between remote work groups. Display a count table, then a horizontal chart of percentages within groups. Label the axes, title, and legend.

**Hint:** Combine the two `crosstab` examples. Keep `remote_work` as the first column and change the second to `wellness_program`. Add `normalize="index"` only to the percentage table; each row should add up to 100%.

**Example to adapt:**

```python
group_counts = pd.crosstab(df_clean["remote_work"], df_clean["benefits"])
display(group_counts)
group_percent = pd.crosstab(
    df_clean["remote_work"], df_clean["benefits"], normalize="index"
) * 100
group_percent.plot.barh(stacked=True)
plt.xlabel("Percentage of responses within each remote work group")
plt.ylabel("Remote work")
plt.title("Benefits and remote work")
plt.legend(title="Benefits response")
plt.show()
```


In [ ]:
# Task: compare wellness_program between remote_work groups.
# Change both references to the second column and the chart labels.
group_counts = pd.crosstab(df_clean["remote_work"], df_clean["benefits"])
display(group_counts)
group_percent = pd.crosstab(
    df_clean["remote_work"], df_clean["benefits"], normalize="index"
) * 100
group_percent.plot.barh(stacked=True)
plt.xlabel("Percentage of responses within each remote work group")
plt.ylabel("Remote work")
plt.title("Benefits and remote work")
plt.legend(title="Benefits response")
plt.show()


### Two more charts to try

Change the column, color, and title in the recipes below.

- `value_counts()` gives counts; `value_counts(normalize=True) * 100` gives percentages.
- `plot.bar(...)` draws vertical bars; `plot.barh(...)` draws horizontal bars.
- For group comparisons, change `stacked=True` to `False` to place bars side by side.

Update the labels after making changes. A chart alone does not prove that remote work caused the differences.


**Recipe 1: vertical bars with percentages.** Change `seek_help`, the color, and the title.


In [ ]:
values = df_clean["seek_help"].value_counts(normalize=True) * 100
values.plot.bar(color="teal")
plt.ylabel("Percentage of responses in the whole table")
plt.title("Resources about seeking help")
plt.xticks(rotation=0)
plt.show()


**Recipe 2: a pie chart.** Change the column and color list. Percentages refer to all responses in `df_clean`.


In [ ]:
values = df_clean["care_options"].value_counts()
values.plot.pie(autopct="%1.0f%%", colors=["#356b8e", "#d98b39", "#72a98f", "#b3a1c9"])
plt.ylabel("")
plt.title("Awareness of care options")
plt.show()


## Part 4: Hiding information in text

We use only **fictional sentences**.

### Regex: searching by pattern

**Regex** describes what a piece of text we want to find looks like. Imagine the command: **"Find something that looks like an email address and hide it."**

```text
Contact john@example.org.  →  Contact [EMAIL].
```

The long expression in `email_pattern` is a ready-made pattern to copy; you do not need to understand every character. It recognizes common email address formats, including the **@** sign.

`re.sub(pattern, replacement, text)` means: **find matching pieces and replace them with the given text**. This example pattern may miss some addresses.


In [ ]:
import re

synthetic_text = "John Smith lives in London. Contact john@example.org or +48 123 456 789."
email_pattern = r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"

masked_text = re.sub(email_pattern, "[EMAIL]", synthetic_text)
print("Before:", synthetic_text)
print("After: ", masked_text)


For the phone number **+48 123 456 789**, copy this pattern:

```python
phone_pattern = r"\+48 \d{3} \d{3} \d{3}"
```

Read it as: **a plus sign, 48, and three groups of three digits separated by spaces**. `\d{3}` means "three digits". A differently formatted number may not match.


### Exercise 8: hide another type of information

Starting from **masked_text**, replace the phone number with **[PHONE]**. The email address must stay masked. Display the result and check what information is still readable.

**Hint:** Use the phone pattern above. The third argument to `re.sub(...)` should be `masked_text` to preserve the earlier email replacement.

**Example to adapt:**

```python
example = "Meeting code: 123."
re.sub(r"\d{3}", "[CODE]", example)
```


In [ ]:
# Task: mask the phone number in masked_text and display the result.
# Insert the phone pattern and the [PHONE] label.
# Write your own code below.


### Recognizing a person and place from context

A language model also uses the words surrounding a name. In this sentence, it recognizes a person and a city:

```text
John Smith lives in London.
     ↓                  ↓
[PERSON] lives in [PLACE].
```

The model identifies text spans, and the code replaces them with labels. It can make mistakes. Below, we download an English spaCy model, so the example sentence is in English.


In [ ]:
# One-time model setup for the example below.
%pip -q install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
example_text = "John Smith lives in London."
document = nlp(example_text)

# The model returns the detected spans and their categories.
for entity in document.ents:
    print(entity.text, "→", entity.label_)

# PERSON means a person; GPE means a country, city, or region.
labels = {"PERSON": "[PERSON]", "GPE": "[PLACE]"}
semantic_masked = example_text
# Replace from the end so earlier character positions do not shift.
for entity in reversed(document.ents):
    if entity.label_ in labels:
        semantic_masked = (
            semantic_masked[:entity.start_char]
            + labels[entity.label_]
            + semantic_masked[entity.end_char:]
        )
print(semantic_masked)


In [ ]:
# 1. Word similarity — run this cell first.
%pip -q install sentence-transformers

import pandas as pd
from sentence_transformers import SentenceTransformer

similarity_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu",
)

words = ["joy", "satisfaction", "sadness", "gloom"]
word_vectors = similarity_model.encode(words)
word_similarity = similarity_model.similarity(word_vectors, word_vectors).numpy()
pd.DataFrame(word_similarity, index=words, columns=words).round(2)


In [ ]:
# 2. Comment similarity — the same model, whole statements.
# We provide these two reference examples only once.
reference_comments = {
    "positive": "Great workshop, I learned a lot.",
    "negative": "Terrible workshop, I wasted my time.",
}
comments = [
    "Everything was explained clearly and I could practice.",
    "Chaotic explanations, it was hard to keep up.",
    "I am leaving with new skills and energy.",
    "A wasted afternoon, I am even more confused.",
    "Interesting examples, but the pace was too fast.",
]

comment_vectors = similarity_model.encode(comments)
reference_vectors = similarity_model.encode(list(reference_comments.values()))
scores = similarity_model.similarity(comment_vectors, reference_vectors).numpy()
comment_similarity = pd.DataFrame(
    scores, index=comments, columns=list(reference_comments.keys())
)
comment_similarity["Closer example"] = comment_similarity.idxmax(axis=1)
comment_similarity.round(2)


In [32]:
%pip -q install https://github.com/explosion/spacy-models/releases/download/pl_core_news_sm-3.8.0/pl_core_news_sm-3.8.0-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
text = "Jan Kowalski mieszka w Krakowie. Anna Nowak zna Jana Kowalskiego."
document = nlp(text)

people = {}
replacements = []

for entity in document.ents:
    if entity.label_ == "persName":
        key = entity.text.casefold()
        if key not in people:
            people[key] = f"[OSOBA_{len(people) + 1}]"
        label = people[key]
    elif entity.label_ in {"placeName", "geogName"}:
        label = "[MIEJSCE]"
    else:
        continue

    replacements.append((entity.start_char, entity.end_char, label))

masked = text
for start, end, label in reversed(replacements):
    masked = masked[:start] + label + masked[end:]

print(masked)

[OSOBA_1] mieszka w [MIEJSCE]. [OSOBA_2] zna [OSOBA_3].


## Optional extension: Polish names across grammatical cases

The same name can have several forms: **Jan Kowalski**, **Jana Kowalskiego**, and **Janem Kowalskim**. Matching the original spelling gives them different IDs.

Here we use the Polish model's **lemmas**, or base forms, as a shared key. Each full name gets one ID, such as `[OSOBA_1]` (person 1). Reuse the same `person_ids` dictionary across texts to keep the IDs consistent. Places are replaced with `[MIEJSCE]` (place).

Run the setup cell, then the example. The table shows each detected form, its base form, and its assigned ID, so you can inspect the grouping.

**Limits:** this groups detected names, not verified identities. The model can miss a name or predict an incorrect lemma. Two different people with the same full name will share an ID; nicknames, surname-only mentions, and pronouns are not linked automatically. Review the results. All examples below are fictional.


In [35]:
# One-time setup for the Polish examples below.
%pip -q install https://github.com/explosion/spacy-models/releases/download/pl_core_news_sm-3.8.0/pl_core_news_sm-3.8.0-py3-none-any.whl

import spacy
import pandas as pd

# Keep this separate from the English model used earlier.
nlp_pl = spacy.load("pl_core_news_sm")



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [36]:
def mask_polish_people(text, person_ids):
    """Group detected full names by their lemmas; reuse IDs across texts."""
    document = nlp_pl(text)
    replacements = []
    mentions = []

    for entity in document.ents:
        if entity.label_ == "persName":
            base_name = " ".join(
                (token.lemma_ or token.text).casefold()
                for token in entity
                if not token.is_space
            )
            if base_name not in person_ids:
                person_ids[base_name] = f"[OSOBA_{len(person_ids) + 1}]"
            label = person_ids[base_name]
            mentions.append({
                "Detected name": entity.text,
                "Base form": base_name,
                "Person ID": label,
            })
        elif entity.label_ in {"placeName", "geogName"}:
            label = "[MIEJSCE]"
        else:
            continue
        replacements.append((entity.start_char, entity.end_char, label))

    masked = text
    for start, end, label in reversed(replacements):
        masked = masked[:start] + label + masked[end:]

    columns = ["Detected name", "Base form", "Person ID"]
    return masked, pd.DataFrame(mentions, columns=columns)


# One dictionary for both texts: do not reset it inside the loop.
person_ids = {}
polish_examples = [
    "Jan Kowalski mieszka w Krakowie. Anna Nowak zna Jana Kowalskiego.",
    "Rozmawiam z Janem Kowalskim. Odwiedzam Annę Nowak.",
]
mention_tables = []

for example in polish_examples:
    masked, mention_table = mask_polish_people(example, person_ids)
    print(masked)
    mention_tables.append(mention_table)

display(pd.concat(mention_tables, ignore_index=True))


[OSOBA_1] mieszka w [MIEJSCE]. [OSOBA_2] zna [OSOBA_1].
Rozmawiam z [OSOBA_1]. Odwiedzam [OSOBA_2].


,Detected name,Base form,Person ID
0,Jan Kowalski,jan kowalski,[OSOBA_1]
1,Anna Nowak,anna nowak,[OSOBA_2]
2,Jana Kowalskiego,jan kowalski,[OSOBA_1]
3,Janem Kowalskim,jan kowalski,[OSOBA_1]
4,Annę Nowak,anna nowak,[OSOBA_2]
